In [1]:
import Pkg
Pkg.add([
        "CSV",
        "DataFrames",
        "Glob",
        "Makie",
        "CairoMakie"
        ])

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [3]:
import CSV
using DataFrames
using Glob
using Makie, CairoMakie
using Statistics

In [4]:
dfs = [CSV.read(f, DataFrame) for f in glob("*matches.csv", "tennis-data")];

In [5]:
players = vcat([[m.player1; m.player2] for m in dfs]...) |>
    skipmissing |>
    collect |>
    x -> lowercase.(x) |>
    unique |>
    sort |>
    enumerate |>
    x -> reverse.(x) |>
    Dict

Dict{String, Int64} with 1849 entries:
  "mona barthel"       => 1276
  "bjorn fratangelo"   => 284
  "ernests gulbis"     => 569
  "nick kyrgios"       => 1317
  "p. cuevas"          => 1377
  "astra sharma"       => 231
  "m. sakkari"         => 1155
  "t monteiro"         => 1639
  "t. monteiro"        => 1655
  "e bouchard"         => 502
  "regina kulikova"    => 1474
  "borna gojo"         => 293
  "yannick maden"      => 1809
  "kyrian jacquet"     => 994
  "v. zvonareva"       => 1742
  "a. dolgopolov"      => 60
  "alejandro tabilo"   => 125
  "ryan sweeting"      => 1507
  "a. bolt"            => 55
  "timea babos"        => 1694
  "vitalia diatchenko" => 1774
  "alberta brianti"    => 120
  "francoise abanda"   => 629
  "c. taberner"        => 339
  "maxime teixeira"    => 1241
  ⋮                    => ⋮

In [6]:
for m in dfs
    dropmissing!(m, [:player1, :player2])
    transform!(m, :player1 => (names -> getindex.(Ref(players), lowercase.(names))) => :player1id)
    transform!(m, :player2 => (names -> getindex.(Ref(players), lowercase.(names))) => :player2id)
end

In [54]:
df = vcat(dfs...);

In [55]:
players = combine(
    groupby(
        vcat(
            select(df, [:player1id => :id, :player1 => :player]),
            select(df, [:player2id => :id, :player2 => :player])
            ),
        :player
        ),
    [:id => first => :id, :player => unique => :player]
);

In [56]:
### Unify player ids; some players exist as duplicates in the metadata, e.g. C Alcaraz and Carlos Alcaraz

function sel_fullname(name)
    fn = split(name)[1]
    length(fn) == 1 || fn[2] != '.'
end

match_metadata = df

df = players
df.aliases .= Ref(Int[])
fullnames = df[sel_fullname.(df.player),:]
for i in findall(sel_fullname.(df.player))
    fn = lowercase(df[i, :player])
    parts = split(fn)
    abbrev = join([parts[1][1], parts[2:end]...], " ")
    dotabbrev = join([parts[1][1] * ".", parts[2:end]...], " ")
    aliases = []
    append!(aliases, df[lowercase.(df.player) .== abbrev, :id])
    append!(aliases, df[lowercase.(df.player) .== dotabbrev, :id])
    df.aliases[i] = aliases
end

ddf = match_metadata
for player in eachrow(df)
    a = player.aliases
    df = df[.!in.(df.id, Ref(a)), :]
    idx1 = ddf.player1id .∈ Ref(a)
    idx2 = ddf.player2id .∈ Ref(a)
    ddf[idx1,:player1id] .= player.id
    ddf[idx2,:player2id] .= player.id
end

players = select!(df, Not(:aliases))
df = match_metadata;

In [57]:
for m in groupby(df, [:year, :slam])
    filename = "$(m.year[1])-$(m.slam[1])-matches.csv"
    CSV.write(joinpath("clean-data",filename), m)
end

In [58]:
CSV.write(joinpath("clean-data", "players.csv"), players)

"clean-data/players.csv"

In [66]:
dropmissing(df[df.player1id .== 859 .|| df.player2id .== 859,[:event_name]], :event_name)

Row,event_name
,String15
